# GMW v4112 change stats — diagonal extraction

Each source file is a per-location gain/loss matrix: rows are the *start* year (1985–2024), columns are the *end* year (1986–2025), and cell `[start_year, end_year]` holds the cumulative area value for that year pair.

This notebook extracts the diagonal (`start_year = end_year - 1`, i.e. the year-over-year value) from every file and combines them into long tables:

- **Countries** → `countries_change_df`: `location_code`, `end_year`, `value`, `change_type`
- **WDPA protected areas** → `wdpa_change_df`: `wdpa_id`, `end_year`, `value`, `change_type`

In both cases only `*_area.parquet` files are used (the `.xlsx` files are exact duplicates and `_lower`/`_upper` are confidence-interval variants, out of scope here).

In [2]:
import re
from pathlib import Path

import pandas as pd
import geopandas as gpd

## Shared helper

Both datasets share the same matrix layout, so a single function parses the filename (via a dataset-specific regex with named groups `id` and `change_type`) and extracts the diagonal.

In [3]:
def extract_diagonal(path: Path, filename_pattern: re.Pattern, id_col: str) -> pd.DataFrame:
    match = filename_pattern.match(path.name)
    if match is None:
        raise ValueError(f"Filename does not match expected pattern: {path.name}")
    identifier, change_type = match["id"], match["change_type"]

    matrix = pd.read_parquet(path)
    end_years = matrix.columns
    start_years = end_years - 1
    values = matrix.to_numpy()[matrix.index.get_indexer(start_years), matrix.columns.get_indexer(end_years)]

    return pd.DataFrame(
        {
            id_col: identifier,
            "end_year": end_years,
            "value": values,
            "change_type": change_type,
        }
    )

## Countries

In [4]:
COUNTRY_INPUT_DIR = Path("../../../data/GMW_extent_v4/gmw_mng_chng_stats_v4112_corrected")
OUTPUT_DIR = Path("../../../data/GMW_extent_v4")

# location_code is kept verbatim: most are 3-letter ISO codes, but some sub-territories
# use an underscore-joined suffix (e.g. MUS_CA, BES_B, ATF_EI) which the greedy `.+` preserves.
COUNTRY_FILENAME_PATTERN = re.compile(
    r"^(?P<id>.+)_(?P<change_type>gains|losses)_gmw_v4112_gmw_cntry_def_corr_area\.parquet$"
)

In [5]:
# sanity check on a couple of edge-case filenames (sub-territory codes with underscores)
assert COUNTRY_FILENAME_PATTERN.match("SEN_gains_gmw_v4112_gmw_cntry_def_corr_area.parquet")["id"] == "SEN"
assert COUNTRY_FILENAME_PATTERN.match("MUS_CA_losses_gmw_v4112_gmw_cntry_def_corr_area.parquet")["id"] == "MUS_CA"
assert COUNTRY_FILENAME_PATTERN.match("ATF_EI_gains_gmw_v4112_gmw_cntry_def_corr_area.parquet")["id"] == "ATF_EI"

### Discover files

`*_area.parquet` excludes the `_area_lower.parquet` / `_area_upper.parquet` variants by construction (the glob requires `_area` to be immediately followed by the extension).

In [6]:
country_files = sorted(COUNTRY_INPUT_DIR.glob("*_area.parquet"))
print(f"Found {len(country_files)} files")
country_files[:5]

Found 254 files


[PosixPath('../../../data/GMW_extent_v4/gmw_mng_chng_stats_v4112_corrected/ABW_gains_gmw_v4112_gmw_cntry_def_corr_area.parquet'),
 PosixPath('../../../data/GMW_extent_v4/gmw_mng_chng_stats_v4112_corrected/ABW_losses_gmw_v4112_gmw_cntry_def_corr_area.parquet'),
 PosixPath('../../../data/GMW_extent_v4/gmw_mng_chng_stats_v4112_corrected/AGO_gains_gmw_v4112_gmw_cntry_def_corr_area.parquet'),
 PosixPath('../../../data/GMW_extent_v4/gmw_mng_chng_stats_v4112_corrected/AGO_losses_gmw_v4112_gmw_cntry_def_corr_area.parquet'),
 PosixPath('../../../data/GMW_extent_v4/gmw_mng_chng_stats_v4112_corrected/AIA_gains_gmw_v4112_gmw_cntry_def_corr_area.parquet')]

### Extract and combine

In [7]:
countries_change_df = pd.concat(
    (extract_diagonal(path, COUNTRY_FILENAME_PATTERN, "location_code") for path in country_files),
    ignore_index=True,
)
countries_change_df = countries_change_df.sort_values(
    ["location_code", "change_type", "end_year"]
).reset_index(drop=True)
countries_change_df.head()

,location_code,end_year,value,change_type
0,ABW,1986,0.508297,gains
1,ABW,1987,0.430097,gains
2,ABW,1988,0.000000,gains
3,ABW,1989,0.850419,gains
4,ABW,1990,1.026368,gains


### Sanity checks

In [8]:
n_locations = countries_change_df["location_code"].nunique()
n_change_types = countries_change_df["change_type"].nunique()
n_years = countries_change_df["end_year"].nunique()

print(f"locations: {n_locations}")
print(f"change types: {sorted(countries_change_df['change_type'].unique())}")
print(f"end_year range: {countries_change_df['end_year'].min()}\u2013{countries_change_df['end_year'].max()}")

assert len(country_files) == n_locations * n_change_types, "one file expected per (location, change_type)"
assert len(countries_change_df) == len(country_files) * n_years, "one row expected per (location, change_type, end_year)"
assert countries_change_df["value"].isna().sum() == 0, "unexpected missing values in the diagonal"
assert not countries_change_df.duplicated(["location_code", "change_type", "end_year"]).any(), "unexpected duplicate rows"

countries_change_df.describe(include="all")

locations: 127
change types: ['gains', 'losses']
end_year range: 1986–2025


,location_code,end_year,value,change_type
count,10160,10160.000000,10160.000000,10160
unique,127,NaN,NaN,2
top,ABW,NaN,NaN,gains
freq,80,NaN,NaN,5080
mean,NaN,2005.500000,697.382950,NaN
std,NaN,11.543965,2934.656056,NaN
min,NaN,1986.000000,0.000000,NaN
25%,NaN,1995.750000,0.762445,NaN
50%,NaN,2005.500000,21.275141,NaN
75%,NaN,2015.250000,304.286328,NaN


## WDPA protected areas

Same matrix layout as the countries dataset, but the filename's leading token is a numeric WDPA id rather than an ISO code — kept as `wdpa_id` (cast to `int`).

In [9]:
WDPA_INPUT_DIR = Path(
    "../../../data/GMW_extent_v4/mng_chng_stats_v4112_protected_areas/wdpa_jun2026_uid_stats_corr_area"
)

WDPA_FILENAME_PATTERN = re.compile(
    r"^(?P<id>\d+)_(?P<change_type>gains|losses)_wdpa_jun2026_uid_corr_area\.parquet$"
)

### Discover files

This directory holds ~8k relevant `_area.parquet` files (plus their `_lower`/`_upper`/`.xlsx` counterparts), so this section takes noticeably longer to run than the countries one.

In [10]:
wdpa_files = sorted(WDPA_INPUT_DIR.glob("*_area.parquet"))
print(f"Found {len(wdpa_files)} files")
wdpa_files[:5]

Found 8026 files


[PosixPath('../../../data/GMW_extent_v4/mng_chng_stats_v4112_protected_areas/wdpa_jun2026_uid_stats_corr_area/1000_gains_wdpa_jun2026_uid_corr_area.parquet'),
 PosixPath('../../../data/GMW_extent_v4/mng_chng_stats_v4112_protected_areas/wdpa_jun2026_uid_stats_corr_area/1000_losses_wdpa_jun2026_uid_corr_area.parquet'),
 PosixPath('../../../data/GMW_extent_v4/mng_chng_stats_v4112_protected_areas/wdpa_jun2026_uid_stats_corr_area/1001_gains_wdpa_jun2026_uid_corr_area.parquet'),
 PosixPath('../../../data/GMW_extent_v4/mng_chng_stats_v4112_protected_areas/wdpa_jun2026_uid_stats_corr_area/1001_losses_wdpa_jun2026_uid_corr_area.parquet'),
 PosixPath('../../../data/GMW_extent_v4/mng_chng_stats_v4112_protected_areas/wdpa_jun2026_uid_stats_corr_area/1003_gains_wdpa_jun2026_uid_corr_area.parquet')]

### Extract and combine

In [11]:
wdpa_change_df = pd.concat(
    (extract_diagonal(path, WDPA_FILENAME_PATTERN, "wdpa_id") for path in wdpa_files),
    ignore_index=True,
)
wdpa_change_df["wdpa_id"] = wdpa_change_df["wdpa_id"].astype(int)
wdpa_change_df = wdpa_change_df.sort_values(
    ["wdpa_id", "change_type", "end_year"]
).reset_index(drop=True)
wdpa_change_df.head()

,wdpa_id,end_year,value,change_type
0,1,1986,0.000000,gains
1,1,1987,6.043581,gains
2,1,1988,6.205189,gains
3,1,1989,0.000000,gains
4,1,1990,1.224800,gains


In [12]:
LAYER_PATH = Path("../../../data/GMW_extent_v4/gmw_v4112_stats_wdpa_wdoecms/WDPA_poly_Jun2026_intersect_hab_mask_v29_uid_n_pxls_vld.gpkg")
wdpa_layer = gpd.read_file(LAYER_PATH)
wdpa_layer = wdpa_layer[["SITE_ID", "uid"]]
wdpa_layer.head(1)

,SITE_ID,uid
0,24,1


In [13]:
wdpa_change_df_id = wdpa_change_df.merge(
    wdpa_layer,
    how="left",
    left_on="wdpa_id",
    right_on="uid"
)

print(len(wdpa_change_df_id[wdpa_change_df_id["SITE_ID"].isna()]), "rows with missing SITE_ID after merge")
wdpa_change_df_id = wdpa_change_df_id.drop(columns=["uid", "wdpa_id"])
wdpa_change_df_id.head(3)

0 rows with missing SITE_ID after merge


,end_year,value,change_type,SITE_ID
0,1986,0.000000,gains,24
1,1987,6.043581,gains,24
2,1988,6.205189,gains,24


## Add locations IDs and combine

### Countries

In [14]:
locations_combined = pd.read_csv('https://storage.googleapis.com/mangrove_atlas/widget_data/locations_merged.csv')
locations_country = locations_combined[locations_combined['type'] == 'country']
locations_wdpa = locations_combined[locations_combined['type'] == 'wdpa']
locations_wdpa['wdpaid'] = locations_wdpa['wdpaid'].astype(int)
locations_combined.columns


Index(['name', 'iso', 'wdpaid', 'type', 'location_idn', 'id'], dtype='str')

In [15]:
countries_change_df['location_code'] = countries_change_df['location_code'].str.replace('BES_B', 'BES').str.replace('ATF_EI', 'ATF')

country_stats = countries_change_df.merge(
    locations_country[["iso", "id"]],
    how="left",
    left_on="location_code",
    right_on="iso"
)
country_stats.head(3)

,location_code,end_year,value,change_type,iso,id
0,ABW,1986,0.508297,gains,ABW,2707.0
1,ABW,1987,0.430097,gains,ABW,2707.0
2,ABW,1988,0.000000,gains,ABW,2707.0


In [16]:
print(len(country_stats[country_stats["id"].isna()]), "rows with missing location ID after merge")
country_stats[country_stats["id"].isna()]['location_code'].unique()

400 rows with missing location ID after merge


<ArrowStringArray>
['BLM', 'CCK', 'MNP', 'MUS_CA', 'NRU']
Length: 5, dtype: str

In [17]:
country_stats = country_stats.drop(columns=["iso", "location_code"])
country_stats.rename(columns={"id": "location_id", "end_year":"year", "change_type":"indicator"}, inplace=True)
country_stats.dropna(subset=["location_id"], inplace=True)
country_stats = country_stats[["location_id", "year", "indicator", "value"]]
country_stats = country_stats.sort_values(["location_id", "indicator", "year"]).reset_index(drop=True)
country_stats.head()

,location_id,year,indicator,value
0,1645.0,1986,gains,0.0
1,1645.0,1987,gains,0.0
2,1645.0,1988,gains,0.0
3,1645.0,1989,gains,0.0
4,1645.0,1990,gains,0.0


### WDPAs

In [18]:
wdpa_stats = wdpa_change_df_id.merge(
    locations_wdpa[["wdpaid", "id"]],
    how="left",
    left_on="SITE_ID",
    right_on="wdpaid"
)
wdpa_stats.sample(3)

,end_year,value,change_type,SITE_ID,wdpaid,id
130886,1992,0.000000,gains,392112,NaN,NaN
86016,2002,0.000000,gains,198424,NaN,NaN
195365,1991,0.078539,gains,555586010,555586010.0,3693.0


In [ ]:
print("Protected areas with missing location ID after merge:")
print(len(wdpa_stats[wdpa_stats["id"].isna()]), "rows with missing location ID after merge")
print(len(wdpa_stats[wdpa_stats["id"].isna()]['SITE_ID'].unique()), "unique SITE_IDs with missing location ID after merge")

87920 rows with missing location ID after merge
1084 unique SITE_IDs with missing location ID after merge


In [20]:
wdpa_stats.drop(columns=["wdpaid", "SITE_ID"], inplace=True)
wdpa_stats.rename(columns={"id": "location_id", "end_year":"year", "change_type":"indicator"}, inplace=True)
wdpa_stats.dropna(subset=["location_id"], inplace=True)
wdpa_stats = wdpa_stats[["location_id", "year", "indicator", "value"]]
wdpa_stats = wdpa_stats.sort_values(["location_id", "indicator", "year"]).reset_index(drop=True)

wdpa_stats.head()

,location_id,year,indicator,value
0,1563.0,1986,gains,0.000000
1,1563.0,1987,gains,2.469862
2,1563.0,1988,gains,2.947798
3,1563.0,1989,gains,2.868256
4,1563.0,1990,gains,4.541340


In [21]:
locations_stats = pd.concat([country_stats, wdpa_stats], ignore_index=True)
#convert from ha to km2
locations_stats["value"] = locations_stats["value"] * 0.01

locations_stats.to_csv(OUTPUT_DIR / "gmw_v4_change_stats.csv", index=False)

## Combining all data.  
Combining Extent + gain/loss data into a single table for upload.  
At this point, change stats (location_stats) need a change: remove column for "indicator", instead have separate columns for "gain" and "loss" values per year.


In [22]:
# Separating columns for "gain" and "loss" values per year.
stats_pivot = locations_stats.pivot_table(
    index=["location_id", "year"],
    columns="indicator",
    values="value",
    aggfunc="first"
).reset_index()
stats_pivot.rename(columns={"gains": "gain", "losses": "loss"}, inplace=True)
stats_pivot.head(1)

indicator,location_id,year,gain,loss
0,1563.0,1986,0.0,0.0


In [23]:
extent_stats = pd.read_csv('https://storage.googleapis.com/mangrove_atlas/widget_data/widget_data_upload_csv/extent_stats_v4.csv')
extent_stats.head(3)

,location_id,indicator,year,value
0,1645,habitat_extent_area,1985,0.247611
1,1645,habitat_extent_area,1986,0.247611
2,1645,habitat_extent_area,1987,0.247611


In [24]:
print(extent_stats['location_id'].nunique(), "unique location IDs in extent stats")
print(locations_stats['location_id'].nunique(), "unique location IDs in change stats")
print("location IDs in extent stats but not in change stats:", set(extent_stats['location_id']) - set(locations_stats['location_id']))

2967 unique location IDs in extent stats
2967 unique location IDs in change stats
location IDs in extent stats but not in change stats: set()


In [25]:
total_stats = extent_stats.merge(
    stats_pivot,
    how="left",
    on=["location_id", "year"]
)
#replace NaN values with 0 for gain and loss columns
total_stats['gain'] = total_stats['gain'].fillna(0)
total_stats['loss'] = total_stats['loss'].fillna(0)

print(len(extent_stats), "rows in extent stats")
print(len(total_stats), "rows in total stats")

124476 rows in extent stats
124476 rows in total stats


In [26]:
total_stats.head(10)

,location_id,indicator,year,value,gain,loss
0,1645,habitat_extent_area,1985,0.247611,0.000000,0.0
1,1645,habitat_extent_area,1986,0.247611,0.000000,0.0
2,1645,habitat_extent_area,1987,0.247611,0.000000,0.0
3,1645,habitat_extent_area,1988,0.247611,0.000000,0.0
4,1645,habitat_extent_area,1989,0.247611,0.000000,0.0
5,1645,habitat_extent_area,1990,0.247611,0.000000,0.0
6,1645,habitat_extent_area,1991,0.247611,0.000000,0.0
7,1645,habitat_extent_area,1992,0.247611,0.000000,0.0
8,1645,habitat_extent_area,1993,0.249183,0.001564,0.0
9,1645,habitat_extent_area,1994,0.424346,0.175167,0.0


In [27]:
total_stats.to_csv(OUTPUT_DIR / "gmw_v4_extent_gain_loss_stats.csv", index=False)